In [1]:
from __future__ import annotations

import json
import random
from pathlib import Path

import ijson

DATA_DIR = Path.cwd()
GRANTNAV_PATH = DATA_DIR / "grantnav-20260630122623.json"
SAMPLE_SIZE = 5
RANDOM_SEED = 42

assert GRANTNAV_PATH.exists(), f"Missing dataset: {GRANTNAV_PATH}"
print(f"Dataset: {GRANTNAV_PATH.name} ({GRANTNAV_PATH.stat().st_size / 1e9:.2f} GB)")

Dataset: grantnav-20260630122623.json (1.07 GB)


In [2]:
def iter_grants(path: Path):
    """Stream grant objects from the top-level `grants` array."""
    with path.open("rb") as f:
        parser = ijson.items(f, "grants.item")
        while True:
            try:
                yield next(parser)
            except StopIteration:
                break
            except ijson.common.IncompleteJSONError as exc:
                print(f"Warning: stopped early — file may be truncated ({exc})")
                break


def reservoir_sample(iterator, k: int, seed: int | None = None) -> tuple[list[dict], int]:
    """Return k random items from an iterator and the total count seen."""
    rng = random.Random(seed)
    sample: list[dict] = []

    for i, item in enumerate(iterator, start=1):
        if len(sample) < k:
            sample.append(item)
            continue

        j = rng.randrange(i)
        if j < k:
            sample[j] = item

    return sample, i


def summarize_grant(grant: dict) -> dict:
    """Pull a few useful fields for quick inspection."""
    recipient = (grant.get("recipientOrganization") or [{}])[0]
    funder = (grant.get("fundingOrganization") or [{}])[0]
    return {
        "id": grant.get("id"),
        "title": grant.get("title"),
        "amountAwarded": grant.get("amountAwarded"),
        "currency": grant.get("currency"),
        "awardDate": grant.get("awardDate"),
        "recipient": recipient.get("name"),
        "funder": funder.get("name"),
        "top_level_keys": sorted(grant.keys()),
    }

In [11]:
print("Top-level fields:")
with GRANTNAV_PATH.open("rb") as f:
    for key, value in ijson.kvitems(f, ""):
        if key == "grants":
            print("  grants: <array — streamed separately>")
            break
        if isinstance(value, str) and len(value) > 120:
            print(f"  {key}: {value[:120]}...")
        else:
            print(f"  {key}: {value}")

first_grant = next(iter_grants(GRANTNAV_PATH))
print("\nFirst grant summary:")
print(json.dumps(summarize_grant(first_grant), indent=2))

Top-level fields:
  license: See dataset/license within each grant. This file also contains OS data © Crown copyright and database right 2016, Royal ...


IncompleteJSONError: parse error: premature EOF
                                       
                     (right here) ------^


In [9]:
sample, total_grants = reservoir_sample(
    iter_grants(GRANTNAV_PATH),
    k=SAMPLE_SIZE,
    seed=RANDOM_SEED,
)

print(f"Scanned {total_grants:,} grants; showing {len(sample)} random samples:\n")

for i, grant in enumerate(sample, start=1):
    print(f"--- Sample {i} ---")
    print(json.dumps(summarize_grant(grant), indent=2))
    print()

                                       
                     (right here) ------^
)
Scanned 136,093 grants; showing 5 random samples:

--- Sample 1 ---
{
  "id": "360G-devoncf-A589292",
  "title": "Grant to Churches Housing Action Team (CHAT) Mid-Devon Ltd",
  "amountAwarded": 12000,
  "currency": "GBP",
  "awardDate": "2021-01-26T00:00:00+00:00",
  "recipient": "Churches Housing Action Team (CHAT) Mid-Devon Ltd",
  "funder": "Devon Community Foundation",
  "top_level_keys": [
    "additional_data",
    "amountAwarded",
    "awardDate",
    "awardDateDateOnly",
    "beneficiaryLocation",
    "currency",
    "dataSource",
    "dataType",
    "dataset",
    "dateModified",
    "description",
    "filename",
    "fundingOrganization",
    "grantProgramme",
    "id",
    "recipientOrganization",
    "simple_grant_type",
    "title",
    "title_and_description"
  ]
}

--- Sample 2 ---
{
  "id": "360G-tnlcomfund-0045510833",
  "title": "One Love Men's Group",
  "amountAwarded": 8250,
  "curr

In [12]:
# Full record for one random sample (useful for schema exploration)
print(json.dumps(sample[0], indent=2)[:4000])

{
  "id": "360G-devoncf-A589292",
  "title": "Grant to Churches Housing Action Team (CHAT) Mid-Devon Ltd",
  "currency": "GBP",
  "awardDate": "2021-01-26T00:00:00+00:00",
  "dataSource": "http://devoncf.com/",
  "description": "Development of local Emergency Food Network (Mid Devon)",
  "dateModified": "2026-04-20T00:00:00+00:00",
  "amountAwarded": 12000,
  "grantProgramme": [
    {
      "title": "Food Network Programme",
      "title_keyword": "Food Network Programme"
    }
  ],
  "beneficiaryLocation": [
    {
      "countryCode": "GB"
    }
  ],
  "fundingOrganization": [
    {
      "id": "GB-CHC-1057923",
      "name": "Devon Community Foundation",
      "id_and_name": "[\"Devon Community Foundation\", \"GB-CHC-1057923\"]"
    }
  ],
  "recipientOrganization": [
    {
      "id": "GB-CHC-1049478",
      "name": "Churches Housing Action Team (CHAT) Mid-Devon Ltd",
      "charityNumber": "1049478",
      "companyNumber": "03096996",
      "id_and_name": "[\"Churches Housing Actio